# LMA Phase 1 — Clean + Scrape + Tokenize (NO re-download)

Data already on Drive:
- **Nepali downloaded:** 10.78 GB cleaned
- **Hindi downloaded:** 9.1 GB C4 raw + 657 MB wiki raw (clean only)
- **Nepali manual:** ~55 MB OCR raw (clean with loose filters)
- **Hindi manual:** 11.6 MB webcrawl raw (clean + add Wikisource)

**Pipeline:**
1. Setup: mount Drive, unzip code, install deps
2. Clean Hindi C4 + wiki (already downloaded)
3. Scrape Hindi Wikisource (API) + Nepali Wikisource (HTML)
4. Clean all manual data
5. Check sizes, cap downloaded to match manual (20% rule)
6. Merge, splits (98/1/1)
7. Train BPE tokenizers (32k vocab each)
8. Encode, measure tokens, stats
9. Tokenization examples for report

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/LMA_Project'
import os; os.makedirs(PROJ, exist_ok=True)
print('Drive mounted at', PROJ)

In [ ]:
import zipfile, os, sys
zip_path = f'{PROJ}/lma_code.zip'
dest = '/content/lma_code'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print(f'Unzipped to {dest}')
else:
    print(f'ERROR: {zip_path} not found!')

In [ ]:
sys.path.insert(0, '/content/lma_code')
import common.clean, common.dedup, common.splits, common.stats, common.collect, common.langfilter
from hindi.tokenizer.tokenizer import Tokenizer as HiTok
from nepali.tokenizer.tokenizer import Tokenizer as NeTok
print('All imports OK')

In [ ]:
!pip install -q regex datasets beautifulsoup4 lxml
print('deps installed')

## 1. Check existing data on Drive

In [ ]:
import os, glob
PROJ = '/content/drive/MyDrive/LMA_Project'
for lang in ['hindi', 'nepali']:
    print(f'\n--- {lang.upper()} ---')
    for subdir in ['raw/downloaded', 'raw/manual', 'processed/clean', 'processed/clean_manual']:
        d = f'{PROJ}/{lang}/{subdir}'
        if not os.path.isdir(d): continue
        total = 0; files = 0
        for root, _, fs in os.walk(d):
            for f in fs:
                if f.endswith('.jsonl'):
                    total += os.path.getsize(os.path.join(root, f)); files += 1
        if files > 0:
            print(f'  {subdir}: {files} files, {total/1e6:.1f} MB')

## 2. Clean Hindi downloaded (C4 + wiki — already on Drive)
Nepali downloaded is already cleaned (10.78 GB) — skip.

In [ ]:
import glob, subprocess, sys, os
existing = glob.glob('/content/drive/MyDrive/LMA_Project/hindi/processed/clean/hindi_clean_*.jsonl')
if len(existing) > 2:
    print(f'Hindi downloaded already cleaned ({len(existing)} shards) — skipping')
else:
    out = '/content/drive/MyDrive/LMA_Project/hindi/processed/clean'
    for f in sorted(glob.glob('/content/drive/MyDrive/LMA_Project/hindi/raw/downloaded/*.jsonl')):
        print('cleaning', os.path.basename(f))
        subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'hindi',
                        '--input', f, '--output-dir', out,
                        '--source-tag', 'downloaded', '--shard-size', '250000',
                        '--min-chars', '80'], cwd='/content/lma_code')
    print('Hindi downloaded cleaning done')

## 3. Scrape Wikisource (manual data for 20% requirement)

**Hindi Wikisource:** API `prop=text` (rendered HTML), cleaned with BeautifulSoup.
**Nepali Wikisource:** API broken, scrape HTML pages directly.

These are **original-language** public-domain texts, NOT translations.

In [ ]:
# Scraper: shared functions
import json, os, re, time, urllib.parse, urllib.request
from bs4 import BeautifulSoup

def api_get(domain, params):
    url = f'https://{domain}/w/api.php?' + urllib.parse.urlencode({**params, 'format': 'json'})
    req = urllib.request.Request(url, headers={'User-Agent': 'LMA-Project/1.0 (educational)'})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode('utf-8'))

def fetch_html(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'LMA-Project/1.0 (educational)'})
    with urllib.request.urlopen(req, timeout=30) as r:
        return r.read().decode('utf-8', errors='replace')

def clean_html(html_text):
    soup = BeautifulSoup(html_text, 'lxml')
    for tag in soup(['script', 'style', 'nav', 'table', 'form', 'input', 'select']):
        tag.decompose()
    content = soup.find('div', id='mw-content-text')
    if content is None:
        content = soup.find('div', class_='mw-parser-output')
    if content is None:
        content = soup.body
    if content is None:
        return ''
    text = content.get_text(separator='\n')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def list_pages_api(domain, max_pages):
    titles = []; apfrom = None
    while len(titles) < max_pages:
        params = {'action': 'query', 'list': 'allpages', 'apnamespace': '0',
                  'aplimit': '500', 'apfilterredir': 'nonredirects'}
        if apfrom: params['apfrom'] = apfrom
        try:
            data = api_get(domain, params)
        except:
            break
        pages = data.get('query', {}).get('allpages', [])
        titles.extend(p['title'] for p in pages)
        cont = data.get('query-continue', data.get('continue', {}))
        apfrom = cont.get('allpages', {}).get('apfrom') or cont.get('apfrom')
        if not apfrom: break
        time.sleep(0.5)
    return titles[:max_pages]

def list_pages_html(domain, max_pages):
    titles = []
    url = f'https://{domain}/wiki/Special:AllPages?namespace=0'
    try:
        html = fetch_html(url)
        soup = BeautifulSoup(html, 'lxml')
        content = soup.find('div', id='mw-content-text') or soup.body
        if content:
            for a in content.find_all('a', href=True):
                href = a['href']
                if href.startswith('/wiki/') and ':' not in href[6:] and 'Special' not in href:
                    title = urllib.parse.unquote(href[6:])
                    if title not in titles:
                        titles.append(title)
    except Exception as e:
        print(f'  HTML listing error: {e}')
    return titles[:max_pages]

def get_page_text_api(domain, title):
    try:
        data = api_get(domain, {'action': 'parse', 'page': title, 'prop': 'text'})
        html = data.get('parse', {}).get('text', {}).get('*', '')
        return clean_html(html) if html else None
    except:
        return None

def get_page_text_html(domain, title):
    try:
        url = f'https://{domain}/wiki/{urllib.parse.quote(title)}'
        html = fetch_html(url)
        return clean_html(html)
    except:
        return None

print('Scraper functions ready.')

In [ ]:
# Scrape HINDI Wikisource (API works)
lang = 'hindi'; domain = 'hi.wikisource.org'
outdir = f'/content/drive/MyDrive/LMA_Project/{lang}/raw/manual/wikisource'
os.makedirs(outdir, exist_ok=True)
print(f'Listing pages from {domain}...')
titles = list_pages_api(domain, 5000)
print(f'Found {len(titles)} pages. Scraping via API prop=text...')
f_out = open(os.path.join(outdir, f'{lang}_wikisource_0000.jsonl'), 'w', encoding='utf-8')
n_out = total_chars = 0; t0 = time.time()
for i, title in enumerate(titles):
    text = get_page_text_api(domain, title)
    if not text or len(text) < 50: continue
    deva = sum(1 for c in text if 0x0900 <= ord(c) <= 0x097F)
    if deva / max(len(text), 1) < 0.15: continue
    rec = {'text': text, 'source': 'manual', 'lang': lang, 'dataset': 'wikisource',
           'title': title, 'n_chars': len(text), 'n_bytes': len(text.encode('utf-8'))}
    f_out.write(json.dumps(rec, ensure_ascii=False) + '\n')
    n_out += 1; total_chars += len(text)
    if (i+1) % 50 == 0:
        print(f'  [{i+1}/{len(titles)}] saved {n_out}, {total_chars/1e6:.1f} MB', flush=True)
    time.sleep(0.3)
f_out.close()
print(f'HINDI Wikisource DONE: {n_out} pages, {total_chars/1e6:.1f} MB')

In [ ]:
# Scrape NEPALI Wikisource (API broken — HTML method)
lang = 'nepali'; domain = 'ne.wikisource.org'
outdir = f'/content/drive/MyDrive/LMA_Project/{lang}/raw/manual/wikisource'
os.makedirs(outdir, exist_ok=True)
print(f'Listing pages from {domain} (HTML method)...')
titles = list_pages_html(domain, 3000)
titles = list(dict.fromkeys(titles))
print(f'Found {len(titles)} unique pages. Scraping HTML...')
f_out = open(os.path.join(outdir, f'{lang}_wikisource_0000.jsonl'), 'w', encoding='utf-8')
n_out = total_chars = 0; t0 = time.time()
for i, title in enumerate(titles):
    text = get_page_text_html(domain, title)
    if not text or len(text) < 50: continue
    deva = sum(1 for c in text if 0x0900 <= ord(c) <= 0x097F)
    if deva / max(len(text), 1) < 0.15: continue
    rec = {'text': text, 'source': 'manual', 'lang': lang, 'dataset': 'wikisource',
           'title': title, 'n_chars': len(text), 'n_bytes': len(text.encode('utf-8'))}
    f_out.write(json.dumps(rec, ensure_ascii=False) + '\n')
    n_out += 1; total_chars += len(text)
    if (i+1) % 50 == 0:
        print(f'  [{i+1}/{len(titles)}] saved {n_out}, {total_chars/1e6:.1f} MB', flush=True)
    time.sleep(0.5)
f_out.close()
print(f'NEPALI Wikisource DONE: {n_out} pages, {total_chars/1e6:.1f} MB')

In [ ]:
# Verify scraped text is genuine Devanagari
import json, glob
for lang in ['hindi', 'nepali']:
    print(f'\n===== {lang.upper()} WIKISOURCE SAMPLE =====')
    files = glob.glob(f'/content/drive/MyDrive/LMA_Project/{lang}/raw/manual/wikisource/*.jsonl')
    if not files: print('  no files'); continue
    with open(files[0]) as f:
        for i, line in enumerate(f):
            if i >= 3: break
            rec = json.loads(line)
            print(f'  title: {rec.get("title", "?")}')
            print(f'  text: {rec["text"][:200]}')
            print()

## 4. Clean all manual data

In [ ]:
# Clean HINDI MANUAL
import glob, subprocess, sys, os, shutil
out = '/content/drive/MyDrive/LMA_Project/hindi/processed/clean_manual'
if os.path.isdir(out): shutil.rmtree(out)
for f in sorted(glob.glob('/content/drive/MyDrive/LMA_Project/hindi/raw/manual/**/*.jsonl', recursive=True)):
    print('cleaning', os.path.basename(f))
    subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'hindi',
                    '--input', f, '--output-dir', out,
                    '--source-tag', 'manual', '--min-chars', '40',
                    '--min-devanagari-ratio', '0.40'], cwd='/content/lma_code')
print('Hindi manual done')

In [ ]:
# Clean NEPALI MANUAL (loose filters for OCR)
import glob, subprocess, sys, os, shutil
out = '/content/drive/MyDrive/LMA_Project/nepali/processed/clean_manual'
if os.path.isdir(out): shutil.rmtree(out)
manual_dirs = [
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/sahityaras/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/cdc/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/national_library/ocr',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/national_library/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/wikisource',
]
for d in manual_dirs:
    if not os.path.isdir(d): continue
    for f in glob.glob(os.path.join(d, '*.jsonl')):
        print('cleaning', os.path.basename(f))
        subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'nepali',
                        '--input', f, '--output-dir', out,
                        '--source-tag', 'manual', '--min-chars', '15',
                        '--min-devanagari-ratio', '0.20'], cwd='/content/lma_code')
print('Nepali manual done')

In [ ]:
# Check all cleaned sizes
import os, glob
PROJ = '/content/drive/MyDrive/LMA_Project'
for lang in ['hindi', 'nepali']:
    print(f'\n===== {lang.upper()} =====')
    for subdir in ['processed/clean', 'processed/clean_manual']:
        d = f'{PROJ}/{lang}/{subdir}'
        if not os.path.isdir(d): continue
        total = 0; files = 0
        for f in glob.glob(os.path.join(d, '*.jsonl')):
            total += os.path.getsize(f); files += 1
        print(f'  {subdir}: {files} files, {total/1e6:.1f} MB')

## 5. Balance: cap downloaded to match manual (20% rule)

In [ ]:
# Measure and plan
import os, glob, json
PROJ = '/content/drive/MyDrive/LMA_Project'
for lang in ['hindi', 'nepali']:
    print(f'\n===== {lang.upper()} =====')
    for label, subdir in [('downloaded', 'processed/clean'), ('manual', 'processed/clean_manual')]:
        d = f'{PROJ}/{lang}/{subdir}'
        if not os.path.isdir(d): continue
        total_chars = 0; n_docs = 0
        for f in glob.glob(os.path.join(d, '*.jsonl')):
            with open(f) as fh:
                for line in fh:
                    try:
                        rec = json.loads(line)
                        total_chars += rec.get('n_chars', len(rec.get('text', '')))
                        n_docs += 1
                    except: pass
        print(f'  {label}: {n_docs:,} docs, {total_chars/1e6:.1f}M chars, ~{total_chars/3.5/1e6:.0f}M tokens')

In [ ]:
# Cap downloaded shards: keep first N to match manual*4
import os, glob, json, shutil
PROJ = '/content/drive/MyDrive/LMA_Project'
for lang in ['hindi', 'nepali']:
    manual_dir = f'{PROJ}/{lang}/processed/clean_manual'
    dl_dir = f'{PROJ}/{lang}/processed/clean'
    capped_dir = f'{PROJ}/{lang}/processed/clean_capped'
    if not os.path.isdir(manual_dir) or not os.path.isdir(dl_dir):
        print(f'{lang}: missing dirs, skipping'); continue
    manual_chars = 0
    for f in glob.glob(f'{manual_dir}/*.jsonl'):
        with open(f) as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                    manual_chars += rec.get('n_chars', len(rec.get('text', '')))
                except: pass
    target_dl = manual_chars * 4
    print(f'\n{lang}: manual={manual_chars/1e6:.1f}M chars, target downloaded={target_dl/1e6:.1f}M chars')
    os.makedirs(capped_dir, exist_ok=True)
    for f in glob.glob(f'{capped_dir}/*.jsonl'):
        os.remove(f)
    accumulated = 0
    for f in sorted(glob.glob(f'{dl_dir}/*.jsonl')):
        if accumulated >= target_dl: break
        dst = os.path.join(capped_dir, os.path.basename(f))
        shutil.copy2(f, dst)
        with open(f) as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                    accumulated += rec.get('n_chars', len(rec.get('text', '')))
                except: pass
        print(f'  copied {os.path.basename(f)} ({accumulated/1e6:.1f}M accumulated)')
    total = manual_chars + accumulated
    if total > 0:
        print(f'{lang}: TOTAL ~{total/1e6:.1f}M chars, manual={manual_chars/total*100:.1f}%, ~{total/3.5/1e6:.0f}M tokens')
    else:
        print(f'{lang}: no data!')

## 6. Merge capped + manual, splits

In [ ]:
import os, glob
for lang in ['hindi', 'nepali']:
    merged = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/merged'
    os.makedirs(merged, exist_ok=True)
    for f in glob.glob(os.path.join(merged, '*.jsonl')):
        os.remove(f)
    for src_dir in [f'/content/drive/MyDrive/LMA_Project/{lang}/processed/clean_capped',
                    f'/content/drive/MyDrive/LMA_Project/{lang}/processed/clean_manual']:
        if os.path.isdir(src_dir):
            for f in glob.glob(os.path.join(src_dir, '*.jsonl')):
                dst = os.path.join(merged, os.path.basename(f))
                if not os.path.exists(dst):
                    os.symlink(os.path.abspath(f), dst)
    print(lang, 'merged:', len(os.listdir(merged)), 'files')

In [ ]:
for lang in ['hindi', 'nepali']:
    inp = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/merged'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/splits'
    !cd /content/lma_code && python -m common.splits --lang {lang} --input {inp} \
        --output-dir {out} --train 0.98 --val 0.01 --test 0.01 --seed 17

## 7. Train BPE tokenizers (32k vocab)

In [ ]:
!cd /content/lma_code && python -m hindi.tokenizer.train --language hindi \
    --data /content/drive/MyDrive/LMA_Project/hindi/splits/hindi_train.jsonl \
    --vocab-size 32000 --char-budget 80000000 \
    --output /content/drive/MyDrive/LMA_Project/hindi/tokenizer/hindi_tokenizer.json

In [ ]:
!cd /content/lma_code && python -m nepali.tokenizer.train --language nepali \
    --data /content/drive/MyDrive/LMA_Project/nepali/splits/nepali_train.jsonl \
    --vocab-size 32000 --char-budget 80000000 \
    --output /content/drive/MyDrive/LMA_Project/nepali/tokenizer/nepali_tokenizer.json

## 8. Encode, measure tokens, stats

In [ ]:
for lang in ['hindi', 'nepali']:
    splits = f'/content/drive/MyDrive/LMA_Project/{lang}/splits'
    tok = f'/content/drive/MyDrive/LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json'
    heldout = f'{splits}/{lang}_heldout.jsonl'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/stats/{lang}_stats.json'
    !cd /content/lma_code && python -m common.stats --lang {lang} --input {splits} \
        --output {out} --tokenizer {tok} --heldout {heldout}

In [ ]:
import json, os
for lang in ['hindi', 'nepali']:
    sp = f'/content/drive/MyDrive/LMA_Project/{lang}/stats/{lang}_stats.json'
    if os.path.exists(sp):
        print(f'\n===== {lang.upper()} STATS =====')
        s = json.load(open(sp))
        for k, v in s.items():
            if k != 'source_breakdown':
                print(f'  {k}: {v}')
        if 'source_breakdown' in s:
            print('  source_breakdown:')
            for kk, vv in s['source_breakdown'].items():
                print(f'    {kk}: {vv}')

## 9. Tokenization examples

In [ ]:
import sys, os
sys.path.insert(0, '/content/lma_code')
from nepali.tokenizer.tokenizer import Tokenizer as NeTok
from hindi.tokenizer.tokenizer import Tokenizer as HiTok
examples = {
  'hindi': ['नमस्ते दुनिया, यह एक परीक्षण वाक्य है।', 'भारत एक विशाल और विविध देश है।'],
  'nepali': ['नमस्ते संसार, यो एक परीक्षण वाक्य हो।', 'नेपाल एक सुन्दर हिमालयको देश हो।'],
}
for lang, toks in [('hindi', HiTok), ('nepali', NeTok)]:
    tpath = f'/content/drive/MyDrive/LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json'
    if not os.path.exists(tpath): print(f'{lang}: tokenizer not found'); continue
    tok = toks.load(tpath)
    print(f'\n===== {lang.upper()} tokenization examples =====')
    for ex in examples[lang]:
        ids = tok.encode(ex)
        print(f'text: {ex}')
        print(f'ids : {ids}')
        print(f'dec : {tok.decode(ids)}')
        print(f'round_trip: {tok.verify_round_trip(ex)}')
        print(f'stats: {tok.tokenization_stats(ex)}')
        print()

## 10. Done!
Copy stats + tokenizer JSONs to repo on your Mac for git.
Paste stats output so I can fill in the Phase 1 report.